# Empirical diagnostic of $\varepsilon_{g,\mathrm{dist}}$ on SDXL-Lightning vs. SDXL-Base

This notebook empirically verifies whether the student model (SDXL-Lightning) and the teacher
model (SDXL-Base) produce similar Jacobians with respect to the conditioning input $x$
(the text/prompt embedding).

Key differences from the Turbo version:
- **Student**: `ByteDance/SDXL-Lightning` (4-step, same-resolution distillation at 1024×1024)
- **Teacher**: `stabilityai/stable-diffusion-xl-base-1.0` (20-step DDIM at 1024×1024)
- Student uses `EulerDiscreteScheduler` (Lightning default), teacher uses `DDIMScheduler`
- CFG: teacher uses 7.5, student uses 0.0 (Lightning was distilled with CFG baked in)

- **$f_\star(x,\eta)$** — Teacher: SDXL-Base, 20 DDIM steps  
- **$f_\phi(x,\eta)$** — Student: SDXL-Lightning, 4 steps  
- **$\varepsilon_s = \|f_\star(x) - f_\phi(x)\|$** — Output gap  
- **$\varepsilon_g \approx \mathbb{E}_v\|\partial_x(v^\top f_\star) - \partial_x(v^\top f_\phi)\|$** — Jacobian gap (via random VJPs)

**Sections**
1. Imports & configuration  
2. Load models  
3. Sample images — visual comparison  
4. Gradient sanity check  
5. LCD-ODE coupled $\varepsilon_s$ / $\varepsilon_g$ experiment  
6. Results: summary table + plots  

## §1 — Imports & configuration

In [ ]:
# ── dependencies ─────────────────────────────────────────────────────────────
# If running fresh:
# !pip install diffusers transformers accelerate -q

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import csv
import warnings
warnings.filterwarnings("ignore")

from diffusers import (
    StableDiffusionXLPipeline,
    EulerDiscreteScheduler,
    DDIMScheduler,
)
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# ── device ───────────────────────────────────────────────────────────────────
SEED   = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Device: {device}  |  dtype: {dtype}")

In [ ]:
# ── experiment hyper-parameters ───────────────────────────────────────────────
# SDXL-Lightning and SDXL-Base are BOTH native 1024×1024 — no resolution mismatch
HEIGHT, WIDTH = 1024, 1024

# image sampling (§3)
N_TEACHER_SAMPLE_STEPS = 20    # DDIM steps for teacher
N_STUDENT_SAMPLE_STEPS = 4     # Lightning steps (1, 2, 4, or 8 are valid)

# §5 experiment
N_RANDOM_V = 10                # random VJP directions per prompt
GUIDANCE_W_TEACHER = 7.5       # standard CFG for SDXL-Base
GUIDANCE_W_STUDENT = 0.0       # CRITICAL — Lightning has CFG distilled in, must be 0

# Lightning checkpoint — ByteDance released separate UNet weights per step count
# Use the 4-step version; options: 1, 2, 4, 8
LIGHTNING_STEPS = 4
LIGHTNING_REPO  = "ByteDance/SDXL-Lightning"
LIGHTNING_CKPT  = f"sdxl_lightning_{LIGHTNING_STEPS}step_unet.safetensors"

STYLE_SUFFIX = ", photorealistic photograph, DSLR, natural lighting, sharp focus"

PROMPTS = [p + STYLE_SUFFIX for p in [
    "a golden retriever playing in a sunny park",
    "a red sports car on a mountain road",
    "a bowl of colorful fruit on a wooden table",
    "a child flying a kite in an open meadow",
    "an astronaut floating in outer space",
]]

# time_ids: [orig_H, orig_W, crop_top, crop_left, target_H, target_W]
# At native 1024 both teacher and student use the same values
time_ids = torch.tensor(
    [[HEIGHT, WIDTH, 0, 0, HEIGHT, WIDTH]],
    dtype=dtype,
    device=device
)

## §2 — Load models

SDXL-Lightning is distributed as **UNet weights only** (a `.safetensors` file).
We load the full SDXL-Base pipeline, then hot-swap the UNet with Lightning weights.
This means teacher and student share the **same VAE and text encoders** exactly —
and both operate at the same native resolution (1024×1024).

In [ ]:
TEACHER_ID = "stabilityai/stable-diffusion-xl-base-1.0"

print("Loading teacher pipeline (SDXL-Base + DDIM scheduler)...")
teacher_pipe = StableDiffusionXLPipeline.from_pretrained(
    TEACHER_ID,
    torch_dtype=dtype,
    variant="fp16" if dtype == torch.float16 else None,
    use_safetensors=True,
).to(device)
teacher_pipe.scheduler = DDIMScheduler.from_config(teacher_pipe.scheduler.config)
teacher_pipe.set_progress_bar_config(disable=True)
print("  ✓ Teacher ready.")

In [ ]:
# ── Load Lightning UNet weights into a copy of the SDXL-Base pipeline ────────
# Lightning is distributed as UNet-only safetensors — we hot-swap the UNet.
# The VAE, text encoders, and tokenizers remain identical to the teacher.
# This is the correct way per ByteDance's official release notes.

print(f"Loading SDXL-Lightning UNet ({LIGHTNING_STEPS}-step checkpoint)...")

# Build a fresh SDXL-Base pipeline to host the Lightning UNet
student_pipe = StableDiffusionXLPipeline.from_pretrained(
    TEACHER_ID,
    torch_dtype=dtype,
    variant="fp16" if dtype == torch.float16 else None,
    use_safetensors=True,
).to(device)

# Download and load Lightning UNet weights
ckpt_path = hf_hub_download(LIGHTNING_REPO, LIGHTNING_CKPT)
student_pipe.unet.load_state_dict(load_file(ckpt_path, device=device))

# Lightning requires EulerDiscreteScheduler with specific trailing timestep spacing
student_pipe.scheduler = EulerDiscreteScheduler.from_config(
    student_pipe.scheduler.config,
    timestep_spacing="trailing",  # CRITICAL for Lightning
)
student_pipe.set_progress_bar_config(disable=True)

student_unet = student_pipe.unet
print("  ✓ Student (SDXL-Lightning) ready.")
print(f"  Note: teacher and student share VAE + text encoders — only UNet differs.")

In [ ]:
teacher_pipe.unet.enable_gradient_checkpointing()
student_unet.enable_gradient_checkpointing()
print("  ✓ Gradient checkpointing enabled for both UNets.")

## §3 — Sample images & visual comparison

Same seed for both models. Any visual difference is purely due to the UNet, not the noise.

In [ ]:
teacher_images, student_images = [], []
print("Sampling images...")
for i, prompt in enumerate(PROMPTS):

    gen = torch.Generator(device=device).manual_seed(SEED + i)
    with torch.no_grad():
        t_img = teacher_pipe(
            prompt,
            num_inference_steps=N_TEACHER_SAMPLE_STEPS,
            guidance_scale=GUIDANCE_W_TEACHER,
            generator=gen,
            height=HEIGHT, width=WIDTH,
        ).images[0]

    gen = torch.Generator(device=device).manual_seed(SEED + i)  # reset same seed
    with torch.no_grad():
        s_img = student_pipe(
            prompt,
            num_inference_steps=N_STUDENT_SAMPLE_STEPS,
            guidance_scale=GUIDANCE_W_STUDENT,  # must be 0 for Lightning
            generator=gen,
            height=HEIGHT, width=WIDTH,
        ).images[0]

    teacher_images.append(t_img)
    student_images.append(s_img)
    print(f"  [{i+1}/{len(PROMPTS)}] {prompt[:60]}")


fig, axes = plt.subplots(2, len(PROMPTS), figsize=(4 * len(PROMPTS), 9))
for i, prompt in enumerate(PROMPTS):
    for row, (img, label) in enumerate([
        (teacher_images[i], f"Teacher\n({N_TEACHER_SAMPLE_STEPS} DDIM steps)"),
        (student_images[i], f"Student Lightning\n({N_STUDENT_SAMPLE_STEPS} steps)"),
    ]):
        axes[row, i].imshow(img)
        axes[row, i].set_title(f'"{prompt[:26]}…"', fontsize=8)
        axes[row, i].axis("off")

axes[0, 0].set_ylabel("Teacher (SDXL-Base)", fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel("Student (SDXL-Lightning)", fontsize=11, fontweight="bold")
plt.suptitle(
    f"Teacher (SDXL-Base, {N_TEACHER_SAMPLE_STEPS} steps) vs "
    f"Student (SDXL-Lightning, {N_STUDENT_SAMPLE_STEPS} steps) — same seed η per prompt\n"
    f"Both at {HEIGHT}×{WIDTH} — no resolution mismatch",
    fontsize=12,
)
plt.tight_layout()
plt.savefig("fig1_samples_lightning.png", dpi=120, bbox_inches="tight")
plt.show()

## §4 — Gradient sanity check

Verify that `torch.autograd` can differentiate through a single UNet forward pass
w.r.t. the text embedding $x$. Both UNets should be differentiable.

In [ ]:
def encode_prompt(pipe, prompt):
    """Return (prompt_embeds, pooled_embeds). No grad attached."""
    with torch.no_grad():
        pe, _, pooled, _ = pipe.encode_prompt(
            prompt=prompt, device=device,
            num_images_per_prompt=1, do_classifier_free_guidance=False,
        )
    return pe.to(dtype), pooled.to(dtype)


def encode_prompt_full(pipe, prompt):
    """Returns positive and negative embeddings for CFG."""
    with torch.no_grad():
        pe, _, pooled, _ = pipe.encode_prompt(
            prompt=prompt, device=device,
            num_images_per_prompt=1, do_classifier_free_guidance=False,
        )
        ne, _, n_pooled, _ = pipe.encode_prompt(
            prompt="", device=device,
            num_images_per_prompt=1, do_classifier_free_guidance=False,
        )
    return pe.to(dtype), pooled.to(dtype), ne.to(dtype), n_pooled.to(dtype)


def unet_forward(unet, latent, prompt_embeds, pooled_embeds, timestep):
    """Single UNet denoising step. Returns predicted noise tensor."""
    added_cond = {
        "text_embeds": pooled_embeds,
        "time_ids": time_ids,  # shared global — correct for both models at 1024
    }
    return unet(
        latent, timestep,
        encoder_hidden_states=prompt_embeds,
        added_cond_kwargs=added_cond,
    ).sample


pe0, pooled0 = encode_prompt(teacher_pipe, PROMPTS[0])

torch.manual_seed(SEED)
eta0 = torch.randn(1, 4, HEIGHT // 8, WIDTH // 8, dtype=dtype, device=device)
t500 = torch.tensor([500], device=device)

print("§4 Gradient sanity check")
print("-" * 60)
for name, unet in [("Teacher", teacher_pipe.unet), ("Student (Lightning)", student_unet)]:
    unet.train()
    x = pe0.detach().requires_grad_(True)
    out = unet_forward(unet, eta0, x, pooled0, t500)
    out.sum().backward()

    g = x.grad
    print(f"  {name:22s} | shape {list(g.shape)} | "
          f"NaN: {torch.isnan(g).any().item()} | "
          f"norm: {g.norm().item():.5f}")

print("\n  ✓ Both UNets are differentiable w.r.t. prompt embeddings.")

## §5 — Coupled $\varepsilon_s$ / $\varepsilon_g$ experiment

We compare the **full denoising**:
- **Teacher** $f_\star(x)$: 20 DDIM steps, CFG=7.5, 1024×1024
- **Student** $f_\phi(x)$: 4 Lightning steps, CFG=0.0, 1024×1024

$$\varepsilon_s = \|f_\star(x) - f_\phi(x)\| \quad\text{(output gap)}$$
$$\varepsilon_g \approx \frac{1}{K}\sum_k \|\partial_x(v_k^\top f_\star) - \partial_x(v_k^\top f_\phi)\| \quad\text{(Jacobian gap via random VJPs)}$$

### Helper functions

In [ ]:
def teacher_full(scheduler_template, z_T, prompt_embeds, pooled_embeds, neg_embeds, n_pooled_embeds):
    scheduler = copy.deepcopy(scheduler_template)
    scheduler.set_timesteps(N_TEACHER_SAMPLE_STEPS, device=device)

    # Teacher does NOT need init_noise_sigma scaling — DDIM handles this internally
    latent = z_T

    for t in scheduler.timesteps:
        noise_pred = teacher_pipe.unet(
            latent, t,
            encoder_hidden_states=prompt_embeds,
            added_cond_kwargs={"text_embeds": pooled_embeds, "time_ids": time_ids},
        ).sample

        if GUIDANCE_W_TEACHER > 0:
            noise_uncond = teacher_pipe.unet(
                latent, t,
                encoder_hidden_states=neg_embeds,
                added_cond_kwargs={"text_embeds": n_pooled_embeds, "time_ids": time_ids},
            ).sample
            noise_pred = (1 + GUIDANCE_W_TEACHER) * noise_pred - GUIDANCE_W_TEACHER * noise_uncond

        latent = scheduler.step(noise_pred, t, latent).prev_sample

    return latent

def student_lightning(scheduler_template, z_T, prompt_embeds, pooled_embeds):
    scheduler = copy.deepcopy(scheduler_template)
    scheduler.set_timesteps(N_STUDENT_SAMPLE_STEPS, device=device)

    # CRITICAL: Lightning needs init_noise_sigma scaling on the input latent
    latent = z_T * scheduler.init_noise_sigma

    added_cond = {
        "text_embeds": pooled_embeds,
        "time_ids": time_ids,
    }

    for t in scheduler.timesteps:
        # scale_model_input is a no-op for Euler but good practice
        latent_input = scheduler.scale_model_input(latent, t)
        noise_pred = student_unet(
            latent_input, t,
            encoder_hidden_states=prompt_embeds,
            added_cond_kwargs=added_cond,
        ).sample
        latent = scheduler.step(noise_pred, t, latent).prev_sample

    return latent


def compute_vjp(fn, x_embed, v):
    """
    VJP via one backward pass. Gradient checkpointing on the UNets
    handles the activation memory — this function itself stays simple.
    We explicitly free the graph after each backward to avoid accumulation
    across the N_RANDOM_V loop iterations.
    """
    x = x_embed.detach().requires_grad_(True)
    out = fn(x)
    (out * v).sum().backward()
    grad = x.grad.detach().clone()
    # Explicitly free to avoid graph accumulation across VJP calls
    del out
    x.grad = None
    torch.cuda.empty_cache()
    return grad

def decode(latent):
    with torch.no_grad():
        # 1. Ensure VAE is in float32 for the duration of the decode
        teacher_pipe.vae.to(torch.float32)

        # 2. Scale and check for NaNs in the latent
        # SDXL latents are scaled by ~0.13025; dividing gets them back to VAE range
        l = latent.to(torch.float32) / teacher_pipe.vae.config.scaling_factor

        if torch.isnan(l).any():
            print("Warning: NaNs detected in latents before VAE decoding!")
            return np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

        # 3. Decode
        img = teacher_pipe.vae.decode(l).sample

        # 4. Switch VAE back to dtype (float16) to save memory for the UNet
        teacher_pipe.vae.to(dtype)

    img = (img / 2 + 0.5).clamp(0, 1)
    img = img.squeeze(0).permute(1, 2, 0).float().detach().cpu().numpy()
    return (img * 255).astype("uint8")


### Prompt bank

In [ ]:
PROMPT_ENTRIES = [

    # ════════════════════════════════════════════════════════════════
    # GROUP 1 — PHOTO
    # ════════════════════════════════════════════════════════════════
    ("photo", "landscape",    "a lone pine tree on a flat snowfield, pale sky, photorealistic DSLR photograph, 85mm"),
    ("photo", "landscape",    "a stone wall crossing a flat green field to the horizon, overcast sky, photorealistic DSLR photograph, 35mm"),
    ("photo", "landscape",    "a single wooden rowing boat on a still grey lake, photorealistic DSLR photograph, 50mm"),
    ("photo", "landscape",    "a simple wooden jetty on a calm loch at dawn, pale mist, photorealistic DSLR photograph, 50mm"),
    ("photo", "landscape",    "a lone dead tree on a flat barren plain, pale overcast sky, photorealistic DSLR photograph, 50mm"),
    ("photo", "architecture", "a red lighthouse on a flat rocky headland, pale grey sky, photorealistic DSLR photograph, 70mm"),
    ("photo", "architecture", "a lone stone chapel on a flat green hillside, pale sky, photorealistic DSLR photograph, 35mm"),
    ("photo", "architecture", "a single red telephone box on an empty village green, photorealistic DSLR photograph, 50mm"),
    ("photo", "architecture", "a white wooden church at the end of a straight empty road, photorealistic DSLR photograph, 35mm"),
    ("photo", "architecture", "a derelict stone tower on a coastal headland, grey sea behind, photorealistic DSLR photograph, 50mm"),
    ("photo", "human",        "a hiker in a dark jacket on an open moorland ridge, flat cloudy sky, photorealistic DSLR photograph, 85mm"),
    ("photo", "human",        "a fisherman in waders standing still in a calm river, photorealistic DSLR photograph, 135mm"),
    ("photo", "human",        "an elderly woman in a grey coat on an empty coastal path, photorealistic DSLR photograph, 85mm"),
    ("photo", "human",        "an old shepherd alone in a bare winter field, pale sky, photorealistic DSLR photograph, 135mm"),
    ("photo", "human",        "a young boy in a red jacket on an empty beach at low tide, photorealistic DSLR photograph, 85mm"),
    ("photo", "object",       "a single clay jug on a dark wooden shelf, soft side light, photorealistic DSLR photograph, 85mm"),
    ("photo", "object",       "a white ceramic bowl on grey linen, flat overhead light, photorealistic DSLR photograph, 85mm"),
    ("photo", "object",       "a worn leather boot on a stone floor, soft window light, photorealistic DSLR photograph, 85mm"),
    ("photo", "animal",       "a golden eagle perched on a fence post, flat grey sky, photorealistic DSLR photograph, 200mm"),
    ("photo", "animal",       "a red fox sitting upright on a frozen lake, flat white horizon, photorealistic DSLR photograph, 135mm"),
    ("photo", "animal",       "a white-tailed deer in early morning mist, grey fog background, photorealistic DSLR photograph, 200mm"),
    ("photo", "animal",       "a hare sitting on a flat snowy field, pale sky, photorealistic DSLR photograph, 200mm"),
    ("photo", "animal",       "a barn cat on a stone doorstep, plain white wall behind, photorealistic DSLR photograph, 85mm"),
    ("photo", "animal",       "a puffin on a grass clifftop, flat grey sea behind, photorealistic DSLR photograph, 400mm"),
    ("photo", "animal",       "a wood mouse on a smooth pebble, plain dark background, photorealistic DSLR photograph, 100mm"),

    # ════════════════════════════════════════════════════════════════
    # GROUP 2 — CINEMATIC
    # ════════════════════════════════════════════════════════════════
    ("cinematic", "human",     "a lone detective at the end of a rain-wet street, single streetlamp, cinematic 35mm film"),
    ("cinematic", "human",     "a woman in a dark coat on a clifftop above the sea, overcast sky, cinematic 35mm film"),
    ("cinematic", "human",     "a lighthouse keeper in the doorway at dusk, single figure, cinematic 35mm film"),
    ("cinematic", "human",     "a fisherman in silhouette at the bow of a small boat at dawn, cinematic 35mm film"),
    ("cinematic", "human",     "a soldier alone on a misty plain, desaturated palette, cinematic 35mm film"),
    ("cinematic", "human",     "an astronaut standing alone on a rocky planet surface, pale sky, cinematic wide shot"),
    ("cinematic", "human",     "a monk standing in the doorway of a stone abbey, pale morning light, cinematic 35mm film"),
    ("cinematic", "human",     "a woman holding an umbrella on an empty cobblestone square at night, cinematic 35mm film"),
    ("cinematic", "human",     "a ranger standing on a ridge overlooking a misty valley, cinematic wide shot"),
    ("cinematic", "human",     "a boy standing at the edge of a pier looking at the sea, cinematic 35mm film"),
    ("cinematic", "landscape", "a lone pine on a granite cliff above a still alpine lake, cinematic wide shot, golden hour"),
    ("cinematic", "landscape", "a single rowing boat on a perfectly calm dark fjord at dusk, cinematic wide shot"),
    ("cinematic", "landscape", "a flat salt lake at blue hour with a single figure, mirror reflection, cinematic wide shot"),
    ("cinematic", "landscape", "a winding empty road through autumn forest, fallen leaves, cinematic 35mm film"),
    ("cinematic", "landscape", "a lighthouse on a rocky island at stormy dusk, cinematic wide shot"),
    ("cinematic", "animal",    "a grey heron still on a flat mudflat at low tide, muted palette, cinematic 300mm"),
    ("cinematic", "animal",    "a black horse alone in a flat winter field, pale sky, cinematic 135mm"),
    ("cinematic", "animal",    "a barn owl in flight against a pale dusk sky, wings spread, cinematic 400mm"),
    ("cinematic", "animal",    "a brown trout just below the surface of a clear shallow river, cinematic macro"),
    ("cinematic", "animal",    "a red deer stag silhouetted on a ridge at sunset, cinematic wide shot"),
    ("cinematic", "architecture", "a decaying stone abbey on a flat hillside at dusk, cinematic 35mm film"),
    ("cinematic", "architecture", "a single iron bridge over a dark river at night, one streetlamp, cinematic 35mm film"),
    ("cinematic", "architecture", "a remote mountain train station in winter, empty platform, cinematic 35mm film"),
    ("cinematic", "architecture", "a ruined Roman aqueduct in a flat plain at golden hour, cinematic wide shot"),
    ("cinematic", "architecture", "a lone windmill on a flat Dutch polder at dusk, cinematic wide shot"),

    # ════════════════════════════════════════════════════════════════
    # GROUP 3 — DARK FANTASY
    # ════════════════════════════════════════════════════════════════
    ("dark fantasy", "simple",   "a lone hooded figure at the edge of a frozen lake, pale sky, dark fantasy oil painting"),
    ("dark fantasy", "simple",   "a skeletal tree on a frozen plain, blood moon, dark fantasy painting"),
    ("dark fantasy", "simple",   "a single black crow on a bare gallows post, flat grey sky, dark fantasy illustration"),
    ("dark fantasy", "simple",   "a young witch holding a single candle, plain stone background, dark fantasy portrait"),
    ("dark fantasy", "simple",   "an old alchemist at a candlelit desk, single figure, dark fantasy portrait"),
    ("dark fantasy", "simple",   "a worn iron lantern in a dark stone archway, dark fantasy painting"),
    ("dark fantasy", "simple",   "a black wolf alone on a snowy ridge, moonlit sky, dark fantasy oil painting"),
    ("dark fantasy", "simple",   "a hooded plague doctor on an empty cobblestone street, dark fantasy illustration"),
    ("dark fantasy", "simple",   "a single standing stone with carved runes on a desolate moor, dark fantasy painting"),
    ("dark fantasy", "simple",   "a knight in dark armour at a crossroads, flat overcast sky, dark fantasy oil painting"),
    ("dark fantasy", "moderate", "a stone fortress on a volcanic cliff with rivers of lava below, dark fantasy painting"),
    ("dark fantasy", "moderate", "a vampire in a dark throne room, single candelabra, dark fantasy portrait"),
    ("dark fantasy", "moderate", "a dragon perched on a ruined castle tower at night, dark fantasy illustration"),
    ("dark fantasy", "moderate", "a cursed swamp at dusk, dead trees, will-o-wisps, dark fantasy landscape"),
    ("dark fantasy", "moderate", "a necromancer in a graveyard at midnight, dark fantasy oil painting"),
    ("dark fantasy", "complex",  "a paladin fighting a horde of demons, hellfire rim lighting, crowded battle, dark fantasy painting"),
    ("dark fantasy", "complex",  "a petrified forest where every tree is a screaming face, blood moon, dark fantasy painting"),
    ("dark fantasy", "complex",  "a corrupted space station overrun with eldritch tentacles, dark fantasy sci-fi"),
    ("dark fantasy", "complex",  "an abstract rift in reality tearing open, void geometry, dark fantasy painting"),
    ("dark fantasy", "complex",  "a dragon emerging from a stormy sea at night, epic scale, dark fantasy illustration"),
    ("dark fantasy", "complex",  "a dark fantasy battle scene with armies of undead crossing a burning bridge"),
    ("dark fantasy", "complex",  "a dark fantasy city at night overrun by shadow creatures, chaotic street scene"),
    ("dark fantasy", "complex",  "a dark fantasy summoning ritual with ten robed figures in a circle"),
    ("dark fantasy", "complex",  "a dark fantasy abstract vortex of screaming souls dissolving into geometric void"),
    ("dark fantasy", "complex",  "a massive dark fantasy colossus made of broken swords striding through a burning city"),

    # ════════════════════════════════════════════════════════════════
    # GROUP 4 — CARTOON (OOD negative control)
    # ════════════════════════════════════════════════════════════════
    ("cartoon", "animal",    "a cheerful cartoon dog wearing a backpack, bold outlines, bright flat colors, cartoon illustration"),
    ("cartoon", "animal",    "a cartoon owl sitting on a branch at night, big round eyes, flat colors, cartoon illustration"),
    ("cartoon", "animal",    "a cartoon cat sleeping on a windowsill, pastel colors, bold outlines, cartoon illustration"),
    ("cartoon", "animal",    "a cartoon fox running through autumn forest, bold outlines, warm flat colors, cartoon illustration"),
    ("cartoon", "animal",    "a cartoon bear fishing in a stream, flat colors, thick outlines, cartoon illustration"),
    ("cartoon", "human",     "a cartoon child flying a kite on a hill, bold outlines, primary colors, cartoon illustration"),
    ("cartoon", "human",     "a cartoon old wizard with a long beard, exaggerated features, flat colors, cartoon illustration"),
    ("cartoon", "human",     "a cartoon pirate captain on a ship deck, bold outlines, bright colors, cartoon illustration"),
    ("cartoon", "human",     "a cartoon astronaut floating in space, geometric planet shapes, flat colors, cartoon illustration"),
    ("cartoon", "human",     "a cartoon chef holding a giant cake, exaggerated smile, bold outlines, cartoon illustration"),
    ("cartoon", "landscape", "a cartoon sunny meadow with oversized flowers and a rainbow, flat colors, cartoon illustration"),
    ("cartoon", "landscape", "a cartoon haunted house on a hill at night, glowing windows, flat colors, cartoon illustration"),
    ("cartoon", "landscape", "a cartoon underwater scene with smiling fish and coral, bright colors, cartoon illustration"),
    ("cartoon", "landscape", "a cartoon desert with a single cactus and blazing sun, flat colors, cartoon illustration"),
    ("cartoon", "landscape", "a cartoon snowy village at Christmas with tiny houses, bright colors, cartoon illustration"),
    ("cartoon", "architecture", "a cartoon castle with round towers and a drawbridge, bright colors, cartoon illustration"),
    ("cartoon", "architecture", "a cartoon treehouse connected by rope bridges, flat colors, cartoon illustration"),
    ("cartoon", "architecture", "a cartoon lighthouse with a smiling face, bold outlines, flat colors, cartoon illustration"),
    ("cartoon", "architecture", "a cartoon city street with buildings that have faces, smiling windows, cartoon illustration"),
    ("cartoon", "architecture", "a cartoon bakery storefront with a giant pretzel sign, warm colors, cartoon illustration"),
    ("cartoon", "abstract",  "a cartoon representation of music as bouncing shapes, bold outlines, cartoon illustration"),
    ("cartoon", "abstract",  "a cartoon rainbow exploding from a cloud, bold outlines, primary colors, cartoon illustration"),
    ("cartoon", "abstract",  "a cartoon map with tiny mountains and sea monsters, bold outlines, cartoon illustration"),
    ("cartoon", "abstract",  "a cartoon planet with a smiling face and colorful rings, flat colors, cartoon illustration"),
    ("cartoon", "abstract",  "a cartoon portal swirling with colorful energy, bold outlines, flat colors, cartoon illustration"),
]

PROMPTS_FULL = [text for _, _, text in PROMPT_ENTRIES]

print(f"Total prompts: {len(PROMPT_ENTRIES)}")
from collections import Counter
for grp, n in Counter(g for g,_,_ in PROMPT_ENTRIES).items():
    print(f"  {grp}: {n}")

### Main experiment loop

In [ ]:
# Pre-build scheduler templates
teacher_sched_template = DDIMScheduler.from_config(teacher_pipe.scheduler.config)
student_sched_template = EulerDiscreteScheduler.from_config(
    student_pipe.scheduler.config,
    timestep_spacing="trailing",
)

# VAE stays eval always — we never backprop through it
teacher_pipe.vae.eval()

# UNets: train() mode required for gradient checkpointing
# We'll switch to eval() only for the visual decode, then back to train()
teacher_pipe.unet.train()
student_unet.train()

CSV_PATH = "results/vjp_results_lightning.csv"
CSV_FIELDS = [
    "prompt_idx", "group", "genre", "prompt",
    "eps_s", "eps_g", "eps_g_median", "eps_g_std",
    "f_phi_norm", "J_phi_mean",
    "vjp_00","vjp_01","vjp_02","vjp_03","vjp_04",
    "vjp_05","vjp_06","vjp_07","vjp_08","vjp_09",
]
with open(CSV_PATH, "w", newline="") as f:
    csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

records = []

print("ε_s / ε_g experiment — SDXL-Base (teacher) vs SDXL-Lightning (student)")
print(f"  Resolution: {HEIGHT}×{WIDTH}")
print(f"  VJP directions: {N_RANDOM_V}")
print(f"  Teacher: {N_TEACHER_SAMPLE_STEPS} DDIM | Student: {N_STUDENT_SAMPLE_STEPS} Euler (trailing)")
print()

for i, (style, genre, prompt) in enumerate(PROMPT_ENTRIES):
    print(f"[{i+1}/{len(PROMPT_ENTRIES)}] [{style}] [{genre}]")

    pe, pooled, ne, n_pooled = encode_prompt_full(teacher_pipe, prompt)
    torch.manual_seed(SEED + i)
    z_T = torch.randn(1, 4, HEIGHT // 8, WIDTH // 8, dtype=dtype, device=device)

    def f_teacher(x):
        return teacher_full(teacher_sched_template, z_T, x, pooled, ne, n_pooled)

    def f_student(x):
        return student_lightning(student_sched_template, z_T, x, pooled)

    # ── visual outputs: temporarily eval so batchnorm/dropout behave ──────────
    teacher_pipe.unet.eval()
    student_unet.eval()
    with torch.no_grad():
        out_t = f_teacher(pe)
        out_s = f_student(pe)
    # MUST go back to train before VJPs — gradient checkpointing requires it
    teacher_pipe.unet.train()
    student_unet.train()

    if i == 0:
        print(f"  out_t: min={out_t.min():.3f} max={out_t.max():.3f} mean={out_t.mean():.3f}")
        print(f"  out_s: min={out_s.min():.3f} max={out_s.max():.3f} mean={out_s.mean():.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(decode(out_t))
    axes[0].set_title(f"Teacher ({N_TEACHER_SAMPLE_STEPS} DDIM)")
    axes[0].axis("off")
    axes[1].imshow(decode(out_s))
    axes[1].set_title(f"Lightning ({N_STUDENT_SAMPLE_STEPS} steps)")
    axes[1].axis("off")
    plt.suptitle(f"[{style}][{genre}] {prompt[:60]}", fontsize=9)
    plt.tight_layout()
    plt.show()

    eps_s      = (out_t - out_s).float().norm().item()
    f_phi_norm = out_s.float().norm().item()
    out_shape  = out_t.shape
    del out_t, out_s
    torch.cuda.empty_cache()

    # ── Jacobian gap via random VJPs — UNets are in train() mode here ─────────
    vjp_norms   = []
    J_phi_norms = []
    for kk in range(N_RANDOM_V):
        torch.manual_seed(4000 + 100 * i + kk)
        v = torch.randn(*out_shape, dtype=dtype, device=device)
        v = v / v.norm()
        g_t = compute_vjp(f_teacher, pe, v)
        g_s = compute_vjp(f_student, pe, v)
        vjp_norms.append((g_t - g_s).float().norm().item())
        J_phi_norms.append(g_s.float().norm().item())
        del g_t, g_s
        torch.cuda.empty_cache()

    eps_g        = float(np.mean(vjp_norms))
    eps_g_median = float(np.median(vjp_norms))
    eps_g_std    = float(np.std(vjp_norms))
    J_phi_mean   = float(np.mean(J_phi_norms))

    print(f"  ε_s={eps_s:7.4f} | ε_g(mean)={eps_g:.4f} | ε_g(median)={eps_g_median:.4f} | "
          f"±{eps_g_std:.4f} | ‖f_φ‖={f_phi_norm:.4f} | ‖∂_x[vᵀf_φ]‖={J_phi_mean:.4f}")
    print(f"  ratio ε_g/ε_s = {eps_g/eps_s:.3f}")

    rec = dict(
        prompt_idx=i, group=style, genre=genre, prompt=prompt,
        eps_s=eps_s, eps_g=eps_g, eps_g_median=eps_g_median,
        eps_g_std=eps_g_std, f_phi_norm=f_phi_norm, J_phi_mean=J_phi_mean,
    )
    for kk, val in enumerate(vjp_norms):
        rec[f"vjp_{kk:02d}"] = val

    records.append(rec)
    with open(CSV_PATH, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(rec)

df = pd.DataFrame(records)
print(f"\nDone. {len(df)} prompts completed.")
try:
    from google.colab import files
    files.download(CSV_PATH)
except ImportError:
    print(f"Results saved to {CSV_PATH}")

## §6 — Results: summary table + plots

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = pd.read_csv('results/vjp_results_lightning.csv')
df['r_g_prompt'] = df['eps_g'] / df['J_phi_mean']
df['r_s_prompt'] = df['eps_s'] / df['f_phi_norm']
df['ratio_prompt'] = df['r_g_prompt'] / df['r_s_prompt']

GROUP_ORDER  = ['photo', 'cinematic', 'dark fantasy', 'cartoon']
GROUP_COLORS = {'photo':'#2196F3', 'cinematic':'#4CAF50',
                'dark fantasy':'#FF9800', 'cartoon':'#F44336'}
GROUP_LABELS = {'photo': 'Photo', 'cinematic': 'Cinematic',
                'dark fantasy': 'Dark fantasy', 'cartoon': 'Cartoon'}

N_BOOTSTRAP = 10_000
RNG_SEED    = 42
CI_LEVEL    = 0.95
rng = np.random.default_rng(RNG_SEED)

# ── Helpers ──────────────────────────────────────────────────────
def group_stats(sub):
    eps_s_rms = np.sqrt(np.mean(sub['eps_s']**2))
    eps_g_rms = np.sqrt(np.mean(sub['eps_g']**2))
    yb        = np.sqrt(np.mean(sub['f_phi_norm']**2))
    Jb        = np.sqrt(np.mean(sub['J_phi_mean']**2))
    rs        = eps_s_rms / yb
    rg        = eps_g_rms / Jb
    return dict(eps_dist=eps_s_rms, eps_g_dist=eps_g_rms,
                y_phi_bar=yb, J_phi_bar=Jb,
                r_s=rs, r_g=rg, ratio=rg/rs)

def bootstrap_ci(sub, stat_fn, n_boot=N_BOOTSTRAP, ci=CI_LEVEL):
    boots = [stat_fn(sub.iloc[rng.integers(0, len(sub), size=len(sub))])
             for _ in range(n_boot)]
    boots = np.array(boots)
    alpha = (1 - ci) / 2
    return np.quantile(boots, alpha, axis=0), np.quantile(boots, 1 - alpha, axis=0)

def iqr_clean(sub):
    q1, q3 = sub['ratio_prompt'].quantile([0.25, 0.75])
    iqr = q3 - q1
    return sub[(sub['ratio_prompt'] >= q1 - 1.5*iqr) &
               (sub['ratio_prompt'] <= q3 + 1.5*iqr)]

def print_stats(sub, label, n_removed=0):
    s  = group_stats(sub)
    lo_rs, hi_rs = bootstrap_ci(sub, lambda d: group_stats(d)['r_s'])
    lo_rg, hi_rg = bootstrap_ci(sub, lambda d: group_stats(d)['r_g'])
    lo_rt, hi_rt = bootstrap_ci(sub, lambda d: group_stats(d)['ratio'])
    tag = f"(removed {n_removed})" if n_removed else ""
    print(f"  [{label}]  N={len(sub)} {tag}")
    print(f"    r_s     = {s['r_s']:.4f}  [{lo_rs:.4f}, {hi_rs:.4f}]  95% CI")
    print(f"    r_g     = {s['r_g']:.4f}  [{lo_rg:.4f}, {hi_rg:.4f}]  95% CI")
    print(f"    r_g/r_s = {s['ratio']:.4f}  [{lo_rt:.4f}, {hi_rt:.4f}]  95% CI")

def running_rms_ratio(sub):
    eps_s = sub['eps_s'].values;  eps_g = sub['eps_g'].values
    f_phi = sub['f_phi_norm'].values;  J_phi = sub['J_phi_mean'].values
    ratios = []
    for k in range(1, len(sub) + 1):
        r_s = np.sqrt(np.mean(eps_s[:k]**2)) / np.sqrt(np.mean(f_phi[:k]**2))
        r_g = np.sqrt(np.mean(eps_g[:k]**2)) / np.sqrt(np.mean(J_phi[:k]**2))
        ratios.append(r_g / r_s if r_s > 0 else np.nan)
    return np.array(ratios)

In [ ]:
# ── Global statistics ────────────────────────────────────────────
g = group_stats(df)
print("=== GLOBAL STATISTICS ===")
print(f"  eps_dist        : {g['eps_dist']:.4f}")
print(f"  eps_g_dist      : {g['eps_g_dist']:.4f}")
print(f"  y_phi_bar       : {g['y_phi_bar']:.4f}")
print(f"  J_phi_bar       : {g['J_phi_bar']:.4f}")
print()
print_stats(df, "All groups — with outliers")
print()
df_clean = iqr_clean(df)
print_stats(df_clean, "All groups — without outliers (IQR)", n_removed=len(df)-len(df_clean))
print()

# ── Per-group statistics ─────────────────────────────────────────
print("=== PER-GROUP STATISTICS — WITH OUTLIERS ===")
for grp in GROUP_ORDER:
    sub = df[df['group'] == grp]
    print(f"\n  {GROUP_LABELS[grp]}")
    print_stats(sub, "with outliers")

print()
print("=== PER-GROUP STATISTICS — WITHOUT OUTLIERS (IQR on ratio_prompt) ===")
for grp in GROUP_ORDER:
    sub_full  = df[df['group'] == grp]
    sub_clean = iqr_clean(sub_full)
    print(f"\n  {GROUP_LABELS[grp]}")
    print_stats(sub_clean, "without outliers", n_removed=len(sub_full)-len(sub_clean))

lo_rs, hi_rs = bootstrap_ci(df, lambda d: group_stats(d)['r_s'])
lo_rg, hi_rg = bootstrap_ci(df, lambda d: group_stats(d)['r_g'])


In [ ]:
# ── Running RMS ratio plot ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 3.5), sharey=True)

# Left: all prompts together
all_sorted = df.sort_values('ratio_prompt', ascending=True).reset_index(drop=True)
xs  = np.arange(1, len(all_sorted) + 1)
run = running_rms_ratio(all_sorted)
ax1.plot(xs, run, color='steelblue', lw=2.2)
ax1.axhline(1, color='k', ls='--', lw=1.2, alpha=0.5, label=r'$\hat{r}_g/\hat{r}_s = 1$')
ax1.set_yscale('log')
ax1.set_xlabel('Number of prompts $k$', fontsize=14)
ax1.set_ylabel(r'Running RMS $\hat{r}_g / \hat{r}_s$', fontsize=14)
ax1.set_xlim(1, len(all_sorted))
ax1.set_yticks([1e0, 1e1, 1e2, 1e3])
ax1.tick_params(axis='both', labelsize=12)
ax1.legend(fontsize=12)
ax1.grid(True, which='major', alpha=0.2)
ax1.set_title('All prompts', fontsize=13)

# Right: one curve per group
for grp in GROUP_ORDER:
    sub = (df[df['group'] == grp]
           .sort_values('r_g_prompt', ascending=True)
           .reset_index(drop=True))
    xs  = np.arange(1, len(sub) + 1)
    run = running_rms_ratio(sub)
    ax2.plot(xs, run, color=GROUP_COLORS[grp], lw=2.2, label=GROUP_LABELS[grp])

ax2.axhline(1, color='k', ls='--', lw=1.2, alpha=0.5)
ax2.set_yscale('log')
ax2.set_xlabel('Number of prompts $k$', fontsize=14)
ax2.set_ylabel('')
ax2.set_xlim(1, 25)
ax2.set_xticks(range(1, 26, 2))
ax2.set_yticks([1e0, 1e1, 1e2, 1e3])
ax2.tick_params(axis='both', labelsize=12, labelleft=True)
ax2.legend(fontsize=12)
ax2.grid(True, which='major', alpha=0.2)
ax2.set_title('Per group', fontsize=13)

plt.tight_layout()
plt.savefig("rms_combined.pdf", bbox_inches='tight', dpi=600)
plt.savefig("rms_combined.png", bbox_inches='tight', dpi=600)
plt.show()
print("Saved rms_combined.pdf and rms_combined.png")

In [ ]:
import pandas as pd, numpy as np

df['ratio_prompt'] = (df['eps_g'] / df['J_phi_mean']) / (df['eps_s'] / df['f_phi_norm'])

for grp in df['group'].unique():
    sub = df[df['group'] == grp]
    q1, q3 = sub['ratio_prompt'].quantile([0.25, 0.75])
    iqr = q3 - q1
    out = sub[(sub['ratio_prompt'] < q1 - 1.5*iqr) | (sub['ratio_prompt'] > q3 + 1.5*iqr)]
    print(f"\n=== {grp.upper()} ({len(out)} outliers) ===")
    for _, row in out.iterrows():
        print(f"  idx={int(row['prompt_idx']):3d}  ratio={row['ratio_prompt']:.2f}  {row['prompt'][:80]}")